# Beginner-Friendly AI Material Classifier

This notebook walks through a simple materials classification project using the UCI Glass Identification dataset and `scikit-learn`.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.material_classifier.dataset import CLASS_LABELS, FEATURE_COLUMNS, load_dataset
from src.material_classifier.training import load_saved_bundle, predict_from_dataframe, train_and_save_model

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")

In [ ]:
glass_df = pd.read_csv(ROOT / "data" / "glass.csv")
features, labels, feature_names, class_names = load_dataset(ROOT / "data" / "glass.csv")

print(f"Dataset shape: {glass_df.shape}")
print("Class labels present:")
for class_id in sorted(labels.unique()):
    print(f"  {class_id}: {CLASS_LABELS[class_id]}")

glass_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

glass_df["Type"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Class Distribution")
axes[0].set_xlabel("Type")
axes[0].set_ylabel("Count")

glass_df[FEATURE_COLUMNS].plot(kind="box", ax=axes[1], rot=45)
axes[1].set_title("Feature Spread")
axes[1].set_ylabel("Value")
plt.tight_layout()
plt.show()

## Train And Compare Models

In [ ]:
summary = train_and_save_model(
    dataset_path=ROOT / "data" / "glass.csv",
    model_path=ROOT / "models" / "best_model.joblib",
    metrics_path=ROOT / "models" / "metrics.json",
    comparison_path=ROOT / "models" / "model_comparison.csv",
    confusion_matrix_path=ROOT / "models" / "confusion_matrix.png",
)
summary

In [ ]:
comparison_df = pd.read_csv(ROOT / "models" / "model_comparison.csv")
comparison_df

In [ ]:
with open(ROOT / "models" / "metrics.json", "r", encoding="utf-8") as file:
    metrics = json.load(file)

report = dict(metrics["classification_report"])
accuracy_value = report.pop("accuracy")
report_df = pd.DataFrame(report).T
report_df.loc["accuracy", "f1-score"] = accuracy_value
report_df

In [ ]:
display(Image(filename=ROOT / "models" / "confusion_matrix.png"))

## Sample Prediction

In [ ]:
bundle = load_saved_bundle(ROOT / "models" / "best_model.joblib")
sample_input = pd.read_csv(ROOT / "data" / "sample_input.csv")
predict_from_dataframe(sample_input, bundle)